# Task 5 — Expert-Level Surgical Communication with Qwen-VL

This notebook is designed to be reproducible. It assumes the project folder has this structure:

```text
surgical_vlm/
├── config.py
├── ds/
│   ├── img/
│   ├── ann/
│   └── meta.json
├── scripts/
│   ├── data/
│   ├── modeling/
│   ├── evaluation/
│   └── prompts/
└── outputs/
```

Pipeline:

```text
CholecSeg8k annotations
→ annotation-grounded metadata
→ literature-guided rule teacher labels
→ Qwen LoRA instruction dataset
→ Qwen zero-shot evaluation
→ small LoRA fine-tuning
→ LoRA inference
→ zero-shot vs LoRA comparison
```

## 1. Mount Google Drive

Run this only in Google Colab. If running locally, skip this cell and set `PROJECT_ROOT` manually in the next cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Set project root

In [ ]:
from pathlib import Path
import os
import sys

# Change this path if your project is stored somewhere else.
PROJECT_ROOT = Path("/content/drive/MyDrive/VL-Foundation-with-Surgeon-Level-Intellect/task2")

assert PROJECT_ROOT.exists(), f"Project folder not found: {PROJECT_ROOT}"

os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Current directory:", Path.cwd())
print("Files:", [p.name for p in PROJECT_ROOT.iterdir()])

## 3. Install dependencies

In [ ]:
!pip install -q torch torchvision transformers accelerate pillow qwen-vl-utils peft bitsandbytes tqdm pandas matplotlib

## 4. Verify GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Data preparation can run on CPU, but Qwen inference/fine-tuning requires GPU.")

## 5. Verify dataset and configuration

In [ ]:
import importlib
import config
importlib.reload(config)

print("Image dir:", config.IMAGE_DIR)
print("Annotation dir:", config.ANNOTATION_DIR)
print("Meta path:", config.META_PATH)

assert config.IMAGE_DIR.exists(), "ds/img not found"
assert config.ANNOTATION_DIR.exists(), "ds/ann not found"
assert config.META_PATH.exists(), "ds/meta.json not found"

print("Dataset verified.")

## 6. Import project modules

In [ ]:
from scripts.data.annotation_builder import ExpertCommunicationAnnotationBuilder
from scripts.data.teacher_labeler import CholecystectomyTeacherLabelBuilder

from scripts.modeling.lora_dataset import LoraDatasetBuilder
from scripts.modeling.zero_shot import QwenZeroShotRunner
from scripts.modeling.lora_train import QwenVLLoraTrainer
from scripts.modeling.lora_inference import QwenLoraInferenceRunner

from scripts.evaluation.evaluator import SurgicalCommunicationEvaluator
from scripts.evaluation.visualization import EvaluationTableBuilder

print("Imports successful.")

## 7. Create output folders

In [ ]:
for path in [
    config.ANNOTATION_OUTPUT_DIR,
    config.ZERO_SHOT_OUTPUT_DIR,
    config.LORA_OUTPUT_DIR,
    config.LORA_ADAPTER_OUTPUT_DIR,
    config.LORA_PREDICTION_OUTPUT_DIR,
    config.EVALUATION_OUTPUT_DIR,
    config.FIGURE_OUTPUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Output folders ready.")

## 8. Build annotation-grounded samples

This converts raw DatasetNinja/CholecSeg8k JSON annotations into frame-level metadata.

In [ ]:
builder = ExpertCommunicationAnnotationBuilder(
    image_dir=config.IMAGE_DIR,
    annotation_dir=config.ANNOTATION_DIR,
    meta_path=config.META_PATH,
    output_path=config.EXPERT_ANNOTATION_PATH,
)

samples = builder.build()

splits = builder.split_by_video_id(
    samples=samples,
    train_path=config.TRAIN_JSON_PATH,
    val_path=config.VAL_JSON_PATH,
    test_path=config.TEST_JSON_PATH,
)

print("Total samples:", len(samples))
print("Train:", len(splits["train"]))
print("Val:", len(splits["val"]))
print("Test:", len(splits["test"]))

## 9. Build literature-guided teacher labels

The teacher labels use only annotation-supported classes. The surgical literature is used only to guide safe communication style.

In [ ]:
teacher_builder = CholecystectomyTeacherLabelBuilder(
    input_path=config.EXPERT_ANNOTATION_PATH,
    output_path=config.TEACHER_ANNOTATION_PATH,
)

teacher_samples = teacher_builder.build()

teacher_splits = ExpertCommunicationAnnotationBuilder.split_by_video_id(
    samples=teacher_samples,
    train_path=config.TRAIN_TEACHER_JSON_PATH,
    val_path=config.VAL_TEACHER_JSON_PATH,
    test_path=config.TEST_TEACHER_JSON_PATH,
)

print("Teacher total:", len(teacher_samples))
print("Teacher train:", len(teacher_splits["train"]))
print("Teacher val:", len(teacher_splits["val"]))
print("Teacher test:", len(teacher_splits["test"]))

## 10. Prepare Qwen LoRA instruction dataset

In [ ]:
for input_path, output_path in [
    (config.TRAIN_TEACHER_JSON_PATH, config.LORA_TRAIN_DATA_PATH),
    (config.VAL_TEACHER_JSON_PATH, config.LORA_VAL_DATA_PATH),
    (config.TEST_TEACHER_JSON_PATH, config.LORA_TEST_DATA_PATH),
]:
    lora_builder = LoraDatasetBuilder(
        input_json_path=input_path,
        output_json_path=output_path,
        project_root=PROJECT_ROOT,
    )
    lora_builder.build()

## 11. Inspect one LoRA training sample

In [ ]:
import json

with open(config.LORA_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    lora_train = json.load(f)

print("LoRA train samples:", len(lora_train))
print(json.dumps(lora_train[0], indent=2, ensure_ascii=False)[:2500])

## 12. Visual sanity check: image + teacher answer

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

with open(config.TEST_TEACHER_JSON_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

sample = test_data[0]
image_path = PROJECT_ROOT / "ds" / "img" / sample["image_filename"]

print("Sample ID:", sample["sample_id"])
print("Image path:", image_path)
print("Visible classes:", sample["visible_classes"])

print("\nTeacher answer:")
print(json.dumps(sample["teacher_answer"], indent=2, ensure_ascii=False))

img = Image.open(image_path)
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis("off")
plt.show()

## 13. Run Qwen zero-shot baseline

Start with 20 test samples. You can reduce this to 5 if GPU memory/time is limited.

In [ ]:
runner = QwenZeroShotRunner(
    model_name=config.QWEN_MODEL_NAME,
    input_json_path=config.TEST_TEACHER_JSON_PATH,
    output_json_path=config.ZERO_SHOT_OUTPUT_PATH,
    project_root=PROJECT_ROOT,
    max_samples=20,
)

zero_shot_predictions = runner.run()

print("Zero-shot predictions:", len(zero_shot_predictions))
print("Saved to:", config.ZERO_SHOT_OUTPUT_PATH)

## 14. Inspect zero-shot outputs

In [ ]:
with open(config.ZERO_SHOT_OUTPUT_PATH, "r", encoding="utf-8") as f:
    zero_preds = json.load(f)

for pred in zero_preds[:2]:
    print("\n" + "=" * 100)
    print("Sample:", pred["sample_id"])
    print("Visible classes:", pred["visible_classes"])

    print("\nTeacher answer:")
    print(json.dumps(pred["teacher_answer"], indent=2, ensure_ascii=False))

    print("\nQwen zero-shot output:")
    print(pred["qwen_zero_shot_output"])

## 15. Evaluate zero-shot baseline

In [ ]:
zero_evaluator = SurgicalCommunicationEvaluator(
    prediction_path=config.ZERO_SHOT_OUTPUT_PATH,
    output_json_path=config.ZERO_SHOT_EVAL_PATH,
    output_csv_path=config.ZERO_SHOT_EVAL_TABLE_PATH,
)

zero_eval_results = zero_evaluator.run()

## 16. Show zero-shot evaluation table

In [ ]:
zero_table = EvaluationTableBuilder(config.ZERO_SHOT_EVAL_PATH)
zero_table.print_markdown_table()

## Zero-shot interpretation

The zero-shot model is expected to follow the JSON structure and often use uncertainty-aware language. However, it may hallucinate tools, anatomy, actions, or procedural phases that are not supported by the CholecSeg8k annotation. This motivates LoRA adaptation.

## 17. Small LoRA fine-tuning prototype

This trains a small QLoRA adapter on a limited subset. Start small for reproducibility:

- `max_train_samples=50`
- `max_val_samples=20`
- 1 epoch inside `scripts/modeling/lora_train.py`

If this runs successfully, you can increase the training subset later.

In [ ]:
trainer = QwenVLLoraTrainer(
    model_name=config.QWEN_MODEL_NAME,
    train_json_path=config.LORA_TRAIN_DATA_PATH,
    val_json_path=config.LORA_VAL_DATA_PATH,
    output_dir=config.LORA_ADAPTER_OUTPUT_DIR,
    project_root=PROJECT_ROOT,
    max_train_samples=50,
    max_val_samples=20,
)

adapter_path = trainer.train()

print("Adapter saved at:", adapter_path)

## 18. Run LoRA inference on the same 20 test samples

In [ ]:
lora_runner = QwenLoraInferenceRunner(
    model_name=config.QWEN_MODEL_NAME,
    adapter_path=config.LORA_ADAPTER_OUTPUT_DIR / "final_adapter",
    input_json_path=config.TEST_TEACHER_JSON_PATH,
    output_json_path=config.LORA_PREDICTION_PATH,
    project_root=PROJECT_ROOT,
    max_samples=20,
)

lora_predictions = lora_runner.run()

print("LoRA predictions:", len(lora_predictions))
print("Saved to:", config.LORA_PREDICTION_PATH)

## 19. Evaluate LoRA model

In [ ]:
lora_evaluator = SurgicalCommunicationEvaluator(
    prediction_path=config.LORA_PREDICTION_PATH,
    output_json_path=config.LORA_EVAL_PATH,
    output_csv_path=config.LORA_EVAL_TABLE_PATH,
)

lora_eval_results = lora_evaluator.run()

## 20. Show LoRA evaluation table

In [ ]:
lora_table = EvaluationTableBuilder(config.LORA_EVAL_PATH)
lora_table.print_markdown_table()

## 21. Compare zero-shot vs LoRA

In [ ]:
import pandas as pd

zero_df = pd.read_csv(config.ZERO_SHOT_EVAL_TABLE_PATH)
lora_df = pd.read_csv(config.LORA_EVAL_TABLE_PATH)

comparison = pd.DataFrame({
    "Metric": [
        "Samples",
        "JSON valid rate",
        "Hallucination-free rate",
        "Phase-safe rate",
        "Uncertainty-present rate",
        "Expert-style rate",
        "Average score"
    ],
    "Zero-shot Qwen": [
        len(zero_df),
        zero_df["json_valid"].mean(),
        zero_df["hallucination_free"].mean(),
        zero_df["phase_safe"].mean(),
        zero_df["uncertainty_present"].mean(),
        zero_df["expert_style"].mean(),
        zero_df["total_score_0_to_5"].mean()
    ],
    "LoRA-Qwen": [
        len(lora_df),
        lora_df["json_valid"].mean(),
        lora_df["hallucination_free"].mean(),
        lora_df["phase_safe"].mean(),
        lora_df["uncertainty_present"].mean(),
        lora_df["expert_style"].mean(),
        lora_df["total_score_0_to_5"].mean()
    ]
})

comparison

## 22. Save comparison table

In [ ]:
comparison_path = config.EVALUATION_OUTPUT_DIR / "zero_shot_vs_lora_comparison.csv"
comparison.to_csv(comparison_path, index=False)

print("Saved comparison to:", comparison_path)

## Final conclusion

This notebook demonstrates a complete prototype for Task 5: Expert-Level Surgical Communication.

The system:

1. reads CholecSeg8k image annotations,
2. creates annotation-grounded frame metadata,
3. generates literature-guided but rule-based expert communication labels,
4. prepares Qwen-compatible LoRA instruction data,
5. evaluates Qwen2.5-VL zero-shot behavior,
6. fine-tunes a small LoRA adapter,
7. evaluates LoRA predictions,
8. compares zero-shot vs LoRA.

The key safety design is that the teacher/reference answers use only annotation-supported visible classes. The surgical literature is used for cautious communication style, not for inventing tools, anatomy, phase, or complications.